In [3]:
import numpy as np
X_train = np.load('../datasets/X_train_new.npy')
y_train = np.load('../datasets/y_train_new.npy')

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

X_train shape: (57788, 256, 37)
y_train shape: (57788,)


In [5]:
X_test = np.load('../datasets/X_test.npy')
y_test = np.load('../datasets/y_test.npy')

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

X_test shape: (5000, 256, 37)
y_test shape: (5000,)


In [6]:
X_val = np.load('../datasets/X_val.npy')
y_val = np.load('../datasets/y_val.npy')

print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)

X_val shape: (5000, 256, 37)
y_val shape: (5000,)


In [7]:
print("Train OCD:", sum(y_train == 1))
print("Train Non-OCD:", sum(y_train == 0))

Train OCD: 20000
Train Non-OCD: 37788


In [8]:
X_ocd = X_train[y_train == 1]
y_ocd = y_train[y_train == 1]

print("OCD samples:", X_ocd.shape)

OCD samples: (20000, 256, 37)


### Step 2: Convert sequential data into fixed-size feature vectors by taking the mean across the time axis

In [9]:
# Take the mean across the time_steps (axis=1) to convert sequential data to fixed-size feature vectors
X_train_flat = np.mean(X_train, axis=1)
X_test_flat = np.mean(X_test, axis=1)
X_val_flat = np.mean(X_val, axis=1)

print('Shape after flattening (X_train_flat):', X_train_flat.shape)
print('Shape after flattening (X_test_flat):', X_test_flat.shape)
print('Shape after flattening (X_val_flat):', X_val_flat.shape)

Shape after flattening (X_train_flat): (57788, 37)
Shape after flattening (X_test_flat): (5000, 37)
Shape after flattening (X_val_flat): (5000, 37)


### Step 3: Apply StandardScaler for feature normalization

In [10]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)
X_val_scaled = scaler.transform(X_val_flat)

print('Shape after scaling (X_train_scaled):', X_train_scaled.shape)
print('Shape after scaling (X_test_scaled):', X_test_scaled.shape)
print('Shape after scaling (X_val_scaled):', X_val_scaled.shape)

Shape after scaling (X_train_scaled): (57788, 37)
Shape after scaling (X_test_scaled): (5000, 37)
Shape after scaling (X_val_scaled): (5000, 37)


### Step 4: Apply PCA to reduce features to 4 components (for 4 qubits)

In [11]:
from sklearn.decomposition import PCA

n_components = 4  # Number of principal components for 4 qubits

# Initialize PCA
pca = PCA(n_components=n_components)

# Fit on scaled training data and transform both training and test data
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
X_val_pca = pca.transform(X_val_scaled)

print('Shape after PCA (X_train_pca):', X_train_pca.shape)
print('Shape after PCA (X_test_pca):', X_test_pca.shape)
print('Shape after PCA (X_val_pca):', X_val_pca.shape)

Shape after PCA (X_train_pca): (57788, 4)
Shape after PCA (X_test_pca): (5000, 4)
Shape after PCA (X_val_pca): (5000, 4)


### Step 5: Handle class imbalance by upsampling the minority class

In [13]:
!pip install imbalanced-learn
from imblearn.over_sampling import RandomOverSampler

# Display class distribution before upsampling
print("Class distribution before upsampling (y_train):")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u}: {c} samples")

# Initialize RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Apply oversampling to the training data
X_train_resampled, y_train_resampled = ros.fit_resample(X_train_pca, y_train)

print('\nShape after upsampling (X_train_resampled):', X_train_resampled.shape)
print('Shape after upsampling (y_train_resampled):', y_train_resampled.shape)

# Display class distribution after upsampling
print("\nClass distribution after upsampling (y_train_resampled):")
unique_resampled, counts_resampled = np.unique(y_train_resampled, return_counts=True)
for u, c in zip(unique_resampled, counts_resampled):
    print(f"Class {u}: {c} samples")

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 877.4 kB/s eta 0:00:14
   - -------------------------------------- 0.5/12.4 MB 877.4 kB/s eta 0:00:14
   -- ------------------------------------- 0.8/12.4 MB 866.7 kB/s eta 0:00:14
   --- ------------------------------------ 1.0/12.4 MB 852.7 kB/s eta 0:00:14
   --- ------------------------------------ 1.0/12.4 MB 852.7 kB/s eta 0:00:14
   ---- ----------------------------------- 1.3/12.4 MB 871.9 kB/s eta 0:00:13
   ----- ---------------------------------- 1.6/12.4 MB 895.5 kB/s eta 0:00:13
   ----- ---------------------------------- 1.8/12.4 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'imblearn'

### Step 6: Install and import required Qiskit and Qiskit Machine Learning libraries

In [14]:
!pip install qiskit==0.45.0 qiskit-machine-learning==0.6.1

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [19 lines of output]
  + C:\Python314\python.exe C:\Users\ASUS\AppData\Local\Temp\pip-install-8slq8oxk\numpy_8e6c67d619a14761ab040f652fd3ecfb\vendored-meson\meson\meson.py setup C:\Users\ASUS\AppData\Local\Temp\pip-install-8slq8oxk\numpy_8e6c67d619a14761ab040f652fd3ecfb C:\Users\ASUS\AppData\Local\Temp\pip-install-8slq8oxk\numpy_8e6c67d619a14761ab040f652fd3ecfb\.mesonpy-v_j3ak1z -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\ASUS\AppData\Local\Temp\pip-install-8slq8oxk\numpy_8e6c67d619a14761ab040f652fd3ecfb\.mesonpy-v_j3ak1z\meson-python-native-file.ini
  The Meson build system
  Version: 1.2.99
  Source dir: C:\Users\ASUS\AppData\Local\Temp\pip-install-8slq8oxk\numpy_8e6c67d619a14761ab040f652fd3ecfb
  Build dir: C:\Users\ASUS\AppData\Local\Temp\pip-install-8slq8oxk\numpy_8e6c67d619a14761ab040f652fd3ecfb\.mesonpy-v_j3ak1z
  Build ty

In [16]:
!pip install qiskit
from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.optimizers import COBYLA

print("All imports working ✅")

  Using cached rustworkx-0.17.1-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.6 MB 1.9 MB/s eta 0:00:05
   --- ------------------------------------ 0.8/8.6 MB 2.2 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/8.6 MB 2.2 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/8.6 MB 1.1 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.6 MB 1.2 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.6 MB 1.2 MB/s eta 0:00:06
   -------- ------------------------------- 1.8/8.6 MB 1.3 MB/s eta 0:00:06
   --------- ------------------------------ 2.1/8.6 MB 1.3 MB/s eta 0:00:06
   ---------- ----------------------------- 2.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'qiskit'

### Step 7: Build a quantum model using ZZFeatureMap, RealAmplitudes, and VQC

In [39]:
# Number of qubits
num_qubits = X_train_resampled.shape[1]

# 1. Feature Map
feature_map = ZZFeatureMap(
    feature_dimension=num_qubits,
    reps=1,
    entanglement='linear'
)

# 2. Ansatz
ansatz = RealAmplitudes(
    num_qubits,
    reps=3
)

# 3. Sampler (FIXED)
sampler = StatevectorSampler()   # ✅ changed

# 4. Optimizer
optimizer = COBYLA(maxiter=100)

# 5. VQC Model
vqc = VQC(
    sampler=sampler,
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer
)

print("Quantum model (VQC) assembled successfully ✅")
print(f"Number of qubits: {num_qubits}")

/tmp/ipykernel_16171/3707469856.py:5: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(
/tmp/ipykernel_16171/3707469856.py:12: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(


Quantum model (VQC) assembled successfully ✅
Number of qubits: 4


### Step 8: Train the model on the processed training data

In [ ]:
# Train the VQC model
print("Starting VQC model training...")
vqc.fit(X_train_resampled, y_train_resampled)
print("VQC model training completed.")

Starting VQC model training...


### Step 9: Make predictions on test data

In [ ]:
# Make predictions on the preprocessed test data
y_pred = vqc.predict(X_test_pca)

print("Predictions on test data completed.")

### Step 10: Evaluate the model using Classification report and Confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Convert predictions to binary (0 or 1) if they are not already
# VQC predict returns probability distributions, so we take argmax to get the class
# Or it might return labels directly based on the version/configuration
# For binary classification, VQC usually outputs 0 or 1 directly for labels if fit correctly.

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Plotting the confusion matrix for better visualization
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for VQC')
plt.show()